# poisoned-imagination: selection-only poisoning of a DINO-WM-style latent world-model planner
**Branch:** `feature/poisoned-imagination` | **Gate:** pre-registered in README.md (frozen 2026-11-08)
**One-Click Execution:** Runtime -> Run all (L4). Clones, installs, mounts Drive (results + HF cache), runs pytest on the GPU
with the real DINOv2-small encoder (aborts if invalid), runs pilot + resumable campaign, prints RESULTS.md, unassigns the GPU.
Re-running after a disconnect resumes from Drive (finished units are skipped).

In [ ]:
# Step 1: Credit Guard (auto-unassign on any error or completion) + Colab Secrets
import os, sys, signal, subprocess
from IPython import get_ipython

def auto_teardown(signum=None, frame=None):
    print("\n[Credit Guard] Error, interrupt, or completion! Flushing Drive and unassigning GPU...")
    try:
        from google.colab import drive
        drive.flush_and_unmount()
    except Exception:
        pass
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

def custom_exc_handler(self, etype, value, tb, tb_offset=None):
    self.showtraceback((etype, value, tb), tb_offset=tb_offset)
    auto_teardown()
    return []

ip = get_ipython()
if ip is not None:
    ip.set_custom_exc((Exception,), custom_exc_handler)

try:
    from google.colab import userdata
    for k in ["HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "hf_token"]:
        try:
            tok = userdata.get(k)
            if tok:
                os.environ["HF_TOKEN"] = tok
                os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
                print(f"[Secrets] Loaded Hugging Face token from Colab Secrets ({k}).")
                break
        except Exception:
            pass
except ImportError:
    pass

print("[Ready] Credit guard active. VM will self-terminate on error or completion.")

In [ ]:
# Step 2: GPU check + clone repository
import torch
assert torch.cuda.is_available(), "Switch Colab runtime to GPU (L4 24GB): Runtime -> Change runtime type -> GPU (L4)"
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB | torch {torch.__version__} | CUDA {torch.version.cuda}")

repo_url = "https://github.com/jaimejarauta/decoupled-delta-net.git"
branch = "feature/poisoned-imagination"
target_dir = "/content/poisoned-imagination"
if not os.path.exists(target_dir):
    subprocess.run(["git", "clone", "-b", branch, repo_url, target_dir], check=True)
else:
    subprocess.run(["git", "-C", target_dir, "fetch", "origin", branch], check=True)
    subprocess.run(["git", "-C", target_dir, "reset", "--hard", f"origin/{branch}"], check=True)
os.chdir(target_dir)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Step 3: Installs (pure Python, no CUDA builds)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if os.environ.get("HF_TOKEN"):
    import huggingface_hub
    huggingface_hub.login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print(subprocess.run([sys.executable, "-c", "import pi, transformers; print('transformers', transformers.__version__, '| device', pi.DEV)"],
                     capture_output=True, text=True, check=True).stdout)

In [ ]:
# Step 4: Mount Drive; results (incl. resume state) and the HF cache (DINOv2-small) live there
from google.colab import drive
drive.mount("/content/drive")
os.environ["PI_RESULTS"] = "/content/drive/MyDrive/moonshot_results/poisoned-imagination"
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["PI_ENCODER"] = "dino"
os.makedirs(os.environ["PI_RESULTS"], exist_ok=True)
print("Results dir:", os.environ["PI_RESULTS"], "| existing:", sorted(os.listdir(os.environ["PI_RESULTS"])))

In [ ]:
# Step 5: Pre-flight on the GPU with the real DINOv2-small encoder: env determinism, physics audit (genuine passes,
# PIX caught), selection genuine + biased, encoder sees position and floor colour, CEM, predictor learns, gate rules,
# tiny end-to-end run. ABORTS (and unassigns) if anything fails.
if subprocess.run([sys.executable, "-m", "pytest", "-v", "-p", "no:cacheprovider", "test_pi.py"]).returncode:
    raise RuntimeError("pytest failed on GPU -> campaign aborted (INVALID)")
print("[Pre-flight] all checks passed on GPU.")

In [ ]:
# Step 6: Pilot (seed 100; stops as INVALID if the clean planner fails, or if projected time > 3 h) -> main campaign
# (seeds 0,1,2; resumable) -> gate analysis
p = subprocess.Popen([sys.executable, "-u", "run_all.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
if p.wait():
    raise RuntimeError(f"run_all.py failed (exit {p.returncode}); re-run the notebook to resume from Drive")

In [ ]:
# Step 7: Results
print(open(os.path.join(os.environ["PI_RESULTS"], "RESULTS.md")).read())

In [ ]:
# Step 8: Flush Drive & unassign GPU
auto_teardown()